In [ ]:
import tensorflow as tf

Multiclass Classification (when one object can have multiple labels/ clases)

In [ ]:
'''
pretend we are a fashion company
build a neural network to classify clothing images into different categories
use fashion_mnist (existing dataset of 60,000 training and 10,000 testing vales)
this already there in tensorflow adn has already been sorted into training and testing

https://github.com/zalandoresearch/fashion-mnist

''';

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

(train_data, train_labels), (test_data, test_labels) = fashion_mnist.load_data()

In [ ]:
'''
Label	Description
0	T-shirt/top
1	Trouser
2	Pullover
3	Dress
4	Coat
5	Sandal
6	Shirt
7	Sneaker
8	Bag
9	Ankle boot
''';

In [ ]:
# show the first training sample
print("Training sample", train_data[0])
print("Training label", train_labels[0])

print("Testing sample", test_data[0])
print("Testing label", test_labels[0])

In [ ]:
# check shape of single example
train_data[0].shape, train_labels[0].shape

In [ ]:
# plot a single sample

import matplotlib.pyplot as plt
plt.imshow(train_data[0])

In [ ]:
# check our the sample's label
train_labels[0].item()

In [ ]:
# create a new list to make the labels human readable

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle Boot"]
len(class_names)

In [ ]:
# Plot an example image and its label

index_of_choice = 17
plt.imshow(train_data[index_of_choice])
plt.title(class_names[train_labels[index_of_choice]])

In [ ]:
# input and output shapes respectively
print(train_data[0].shape)
print(len(class_names))

In [ ]:
'''
Input shape = 28 x 28
Output shape = 10 (one per class of clothing)
Loss function = sparse_categorical_crossentropy
Output layer acivation = Softmax (not Sigmoid)
''';

In [ ]:
model_11 = tf.keras.Sequential([
    # we need to flatten the 2D input of 28*28 into 1D of 784
    # so that the neurons can understand it
    tf.keras.layers.Flatten(input_shape = (28, 28)),

    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
    ])

# one hot encoding is a pre-req for categorical-cross-entropy
'''
Gemini said
Sparse Categorical Cross-Entropy is a specific version of the categorical cross-entropy loss function designed
to save you time and computer memory. It allows you to train a multi-class model using integers (like 0, 1, 2)
as your labels, rather than converting them into one-hot encoded vectors (like [1, 0, 0])
'''
model_11.compile(loss = "SparseCategoricalCrossentropy",
                 optimizer = tf.keras.optimizers.Adam(),
                 metrics = ["accuracy"])

model_11_history = model_11.fit(train_data, train_labels,
                                epochs = 10,
                                validation_data = (test_data, test_labels))

'''
validation here gives us an additional acc score on how the model performs on data
it hasn't seen yet '''

In [ ]:
# check the model summary
model_11.summary()

In [ ]:
# check the min and max values of the training data
train_data.min().item(), train_data.max().item()

Improving performace with normmalization

In [ ]:
# neural nets prefer data to be normalised (values between 0-1)
# we can get our training and testing data between 0 and 1 by dividing by max

train_data_norm = train_data/ 255.0
test_data_norm = test_data/ 255.0

train_data_norm.min().item(), train_data_norm.max().item(), test_data_norm.min().item(), test_data_norm.max().item()


In [ ]:
# build a model on the normalised data

model_12 = tf.keras.Sequential([
    # flatten the inputs into a 1D array
    tf.keras.layers.Flatten(input_shape = (28, 28)),

    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])

model_12.compile(
    loss = "SparseCategoricalCrossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics = ["accuracy"]
)

model_12_history = model_12.fit(train_data_norm, train_labels,
                                epochs = 10,
                                validation_data = (test_data_norm, test_labels))

In [ ]:
'''
# same as model 12 but only 4 neurons in each layer
model_13 = tf.keras.Sequential([
    # flatten the inputs into a 1D array
    tf.keras.layers.Flatten(input_shape = (28, 28)),

    tf.keras.layers.Dense(4, activation = "relu"),
    tf.keras.layers.Dense(4, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])

model_13.compile(
    loss = "SparseCategoricalCrossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics = ["accuracy"]
)

model_13_history = model_13.fit(train_data_norm, train_labels,
                                epochs = 10,
                                validation_data = (test_data_norm, test_labels))

''';

In [ ]:
import pandas as pd

# plot non-normalized data loss curves
pd.DataFrame(model_11_history.history).plot(title = "Non-normalized data (Model 11)");

# plot normalized data loss curves
pd.DataFrame(model_12_history.history).plot(title = "Normalized data (Model 12)");

# val- validation


In [ ]:
model_11.summary(), model_12.summary()

Finding the ideal learing rate

In [ ]:
model_14 = tf.keras.Sequential([
    # flatten the inputs into a 1D array
    tf.keras.layers.Flatten(input_shape = (28, 28)),

    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])

model_14.compile(
    loss = "SparseCategoricalCrossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics = ["accuracy"])

# create the learing rate call back
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4 * 10**(epoch/20))


model_14_history = model_14.fit(train_data_norm, train_labels,
                                epochs = 40,
                                validation_data = (test_data_norm, test_labels),
                                callbacks = lr_scheduler,
                                verbose = 1)

In [ ]:
# plot the learning rate decay curve

import numpy as np
import matplotlib.pyplot as plt

lrs  = 1e-3 * (10**(tf.range(40)/20))
plt.semilogx(lrs, model_14_history.history["loss"])
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.title("Learning Rate vs Loss")

'''
For ideal training rate, we look for the point where the loss is still actively
decreasing and is at its lowest before it starts to explode or become unstable again.
Pick a value that is half a step or one step to the left of the absolute minimum.
'''

In [ ]:
# Refit the model with ideal learning rate
# ideal lr = 0.01 (from plot above)


model_15 = tf.keras.Sequential([
    # flatten the inputs into a 1D array
    tf.keras.layers.Flatten(input_shape = (28, 28)),

    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])

model_15.compile(
    loss = "SparseCategoricalCrossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics = ["accuracy"]
)

model_15_history = model_15.fit(train_data_norm, train_labels,
                                epochs = 10,
                                validation_data = (test_data_norm, test_labels))

In [ ]:
pd.DataFrame(model_12_history.history).plot(title = "Model 12");
pd.DataFrame(model_15_history.history).plot(title = "Model 15");


Creating the confusion matrix

In [ ]:
y_preds = model_15.predict(test_data_norm)
y_preds[0]

Prettyfing our confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# 1. Get the predictions
# Use argmax to get the index of the highest probability
y_probs = model_15.predict(test_data)
y_preds = y_probs.argmax(axis=1)

# 2. Create the Figure and Axes first
# This gives you full control over the size for your report
fig, ax = plt.subplots(figsize=(12, 12))

# 3. Plot the Confusion Matrix
# We use 'from_predictions' to do the math and the drawing in one go
disp = ConfusionMatrixDisplay.from_predictions(
    y_true=test_labels,
    y_pred=y_preds,
    display_labels=class_names, # Uses your ['T-shirt/top', 'Trouser', ...] list
    cmap=plt.cm.Blues,
    xticks_rotation=45,
    values_format='d',          # 'd' for decimal integers
    ax=ax                       # Tell it to draw on our pre-defined axes
)

# 4. Final Touch
ax.set_title("Fashion MNIST Confusion Matrix", size=18)
plt.show()

What patterns is our model learning?

In [ ]:
# find layers of our most recent model

model_15.layers

In [ ]:
# extract a layer
model_14.layers[1]

In [ ]:
# get the patterns of a layer in our networks

weights, biases = model_14.layers[1].get_weights()
weights, weights.shape


In [ ]:
# check our bias vector

biases, biases.shape

In [ ]:
# another way to view our deep learning model

from tensorflow.keras.utils import plot_model
# see the inputs and outputs of each layer
plot_model(model_14, show_shapes = True)